# Cylinder 2D with Synthetic Jets

## Referece problem for flow control

A cylinder in a stream at $Re = 100$ sheds vortices, alternately from the top and from the bottom, one pair every $5.8$ time units. That shedding is where most of its drag comes from. Two slots in the surface, one at the top and one at the bottom, blow and suck in antiphase, adding no net mass to the flow; used at the right moments they break the shedding cycle.

The force the flow puts on the cylinder splits into two directions, and both are reported as dimensionless coefficients:

| | direction | uncontrolled value |
|---|---|---|
| drag, $C_d$ | along the stream | around $1.4$, always positive |
| lift, $C_l$ | across the stream | swings between $\pm 0.27$ once per shedding period |

Every $0.4$ time units the 99 probes are handed to the policy, which returns a distribution over $a$. The solver samples it, ramps to it over half the control interval and holds it until the next decision. The reward is $C_{d,0} - (C_d + 0.2\,|C_l|)$ averaged over one shedding period, so a policy that does nothing scores about zero and one that suppresses the shedding scores above it.

This notebook builds the contract that the case (`templates/cylinder2D_jets`) and the policy share, the reward over it, and the training loop. `templates/cylinder2D_jets/README.md` is the long form of the physics and of every change made to the original case.

| symbol | meaning |
|---|---|
| $a$ | the action, the jet velocity, bounded to $[-1, 1]$ against a free stream of $1$ |
| $p_i$ | pressure at 99 probes on an 11 by 9 grid in the wake, the observation |

In [ ]:
%%capture
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

sys.path.insert(0, str(Path.cwd().parent / "src"))

import uqtopus as uqt
import uqtopus.rl as rl
from uqtopus.rl.algos import PPO

case = Path("templates/cylinder2D_jets")
root = Path("experiments/cylinder2D_jets")

have_solver = shutil.which("pimpleFoam") is not None
have_controller = any(
    (Path(os.environ[key]) / "libuqtopusPolicy.so").exists()
    for key in ("FOAM_USER_LIBBIN", "FOAM_LIBBIN")
    if os.environ.get(key)
)

print("pimpleFoam:", "found" if have_solver else "not found")
print("libuqtopusPolicy.so:", "found" if have_controller else "not found")

## 1. A deliberately cheap version of the case

The published case costs about 5 core hours per episode, and 6200 core hours for the training in the paper. What a run costs is the number of cells times the number of time steps, so both were cut:

| | published | here | effect |
|---|---|---|---|
| cells per direction | `meshDensity 1.0` | `meshDensity 0.5` | each cell twice as wide, a quarter as many |
| wake length | 25D | 12D | the far wake fed no probe |
| far field | 9D above and below | 6D | less quiet fluid to solve for |
| **total cells** | **7180** | **1580** | 22 percent |
| **decisions per episode** | **400** | **100** | 40 time units, about 7 shedding periods |

That is roughly eighteen times less work per episode. The 99 probes are unchanged: they cost the solver nothing measurable, and the grid ends at 8D so cutting the domain to 12D removes none of them. One thing to know about them on this mesh: out near 8D the cells are 0.89 wide against a probe spacing of 0.745, so the last columns of probes sample nearly the same fluid and carry less independent information than they did on the published mesh.

`meshDensity 0.5` is the whole of the first cut: half the cells in every direction, so a cell that was 1 by 1 is now 2 by 2. Shortening the domain took one extra step, because the cells grow as they move away from the cylinder and `simpleGrading` states the ratio between the last cell of a block and the first, not the growth per cell. Holding that growth fixed, the 52 cells at ratio 12 that covered 23.75 behind the cylinder become 38 at ratio 6 over 10.75. Keeping the ratio at 12 would have made the cells in the wake twice as wide as intended. `templates/cylinder2D_jets/README.md` works through the arithmetic.

Coarsening this hard shifts the absolute $C_d$ by a few percent, which is why the reward offset is measured on this mesh in section 5 rather than taken from the paper.

In [ ]:
D = 1.0                 # cylinder diameter, unchanged from the published case
jet_width = np.deg2rad(10.0)

x_probe = np.linspace(0.55, 8.0, 11)
y_probe = np.linspace(-1.25, 1.25, 9)
probes = np.array([(x, y, 0.0) for x in x_probe for y in y_probe])

# the mesh parameters are read from the case, so the picture cannot drift from
# what blockMesh will build
blockmesh = (case / "system" / "blockMeshDict").read_text()
density = float(re.search(r"^meshDensity\s+([\d.]+)", blockmesh, re.M).group(1))


def graded(n, ratio):
    """Cell boundaries along one block direction, normalized to [0, 1]."""
    r = ratio ** (1 / (n - 1)) if n > 1 else 1.0
    edges = np.concatenate([[0.0], np.cumsum(r ** np.arange(n))])
    return edges / edges[-1]


def o_grid(mesh_density):
    """Radii and angles of the mesh wrapped around the cylinder."""
    count = lambda base: max(round(base * mesh_density), 1)
    radii = 0.5 * D + graded(count(23), 18) * (1.25 * D - 0.5 * D)
    blocks = [(0, 45, 7, 1.0), (45, 85, 11, 0.25), (85, 95, 6, 1.0),
              (95, 135, 11, 4.0), (135, 180, 7, 1.0)]
    upper = np.unique(np.concatenate(
        [a + graded(count(n), ratio) * (b - a) for a, b, n, ratio in blocks]
    ))
    return radii, np.deg2rad(np.unique(np.concatenate([upper, -upper])))


def wake_cells(length, n, ratio):
    """Cell edges and widths along the block that runs into the wake."""
    r = ratio ** (1 / (n - 1))
    widths = length * (r - 1) / (r**n - 1) * r ** np.arange(n)
    return 1.25 * D + np.concatenate([[0.0], np.cumsum(widths)]), widths


def draw_grid(ax, mesh_density):
    radii, angles = o_grid(mesh_density)
    for r in radii:
        ax.plot(r * np.cos(angles), r * np.sin(angles), color="k", lw=0.4)
    for a in angles:
        ax.plot(radii * np.cos(a), radii * np.sin(a), color="k", lw=0.4)
    slot = np.linspace(np.pi / 2 - jet_width / 2, np.pi / 2 + jet_width / 2, 20)
    for sign in (1, -1):
        ax.plot(0.5 * D * np.cos(slot), sign * 0.5 * D * np.sin(slot),
                color="tab:red", lw=3)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect("equal")


fig, axes = plt.subplot_mosaic([["published", "here"], ["wake", "wake"]],
                               figsize=(10, 7), constrained_layout=True)

draw_grid(axes["published"], 1.0)
axes["published"].set_xlabel("published, 7180 cells")
axes["published"].set_ylabel("y / D")
draw_grid(axes["here"], density)
axes["here"].set_xlabel(f"this case, 1580 cells, same cylinder")

for edges, widths, style, label in (
    wake_cells(23.75, 52, 12) + (dict(color="tab:gray", ls="--"), "published"),
    wake_cells(10.75, 19, 6) + (dict(color="k"), "this case"),
):
    axes["wake"].stairs(widths, edges, baseline=None, label=label, **style)
for x in x_probe:
    axes["wake"].axvline(x, color="tab:blue", lw=0.6, alpha=0.5)
axes["wake"].set_xlabel("x / D, along the wake. The vertical lines are the probe stations")
axes["wake"].set_ylabel("cell width")
axes["wake"].legend(frameon=False, fontsize=9)
plt.show()

published_work, here_work = 7180 * 16000, 1580 * 4000
print(f"cells x time steps per episode: {published_work / 1e6:.0f} M published, "
      f"{here_work / 1e6:.1f} M here, {published_work / here_work:.0f} times less work")
print(f"{len(probes)} probes, {len(x_probe)} by {len(y_probe)}, out to x = {x_probe[-1]:.2f}")

## 2. Policy Spec

One object declares what the policy sees, what it drives and how often. The case dictionary, the ONNX graph and the trajectory file are all derived from it, so they cannot drift apart. `ramp_fraction` is the published ramp of $0.2$ over a control interval of $0.4$.

In [ ]:
CFD_STEP = 0.01
EPISODE_END = 40.0

spec = rl.PolicySpec(
    observation=rl.ObservationSpec(
        sources=(rl.ProbeSource(field_name="p", positions=probes, name="wake"),)
    ),
    action=rl.ActionSpec(
        name="jet",
        targets="jet",
        low=-1.0,
        high=1.0,
        distribution="gaussian",
        ramp_fraction=0.5,
    ),
    control_interval=0.4,
)

n_control = int(EPISODE_END / spec.control_interval)
print(spec)
print(f"{n_control} control steps of {spec.control_interval}, "
      f"{int(spec.control_interval / CFD_STEP)} CFD steps each")

## 3. How the policy gets into OpenFOAM

Python and the solver exchange two files and nothing else. No socket, no shared memory, no process kept alive between them:

| file | direction | content |
|---|---|---|
| `policy.onnx` | Python to solver | the network, frozen, with the contract in its metadata |
| `postProcessing/uqtopusPolicy/<t>/trajectory.dat` | solver to Python | one row per decision: time, the 99 observations, the action applied |

The graph takes two inputs, the observation and one standard normal draw, and returns the action already sampled and already clipped to the bounds. The draw happens in the solver because an ONNX random operator takes its seed as a graph attribute, and all six episodes of one iteration share a single policy file.

ONNX Runtime is the C++ library that evaluates the graph. OpenFOAM does not call it by itself: a boundary condition is a class chosen by the `type` entry, and `uqtopusBoundaryCondition` is not a type that ships with OpenFOAM. It is a class in `uqtopusPolicy/`, compiled once into a library that the stock `pimpleFoam` loads through the `libs` entry of `controlDict`. Section 8 is the build.

Retraining never recompiles anything. Only the `.onnx` file changes.

### The one templated file

`uqtopus.run_simulation` copies the whole case folder to a run directory, renders whatever carries a Jinja2 placeholder, then runs `Allrun` there. Exactly one file of the case has a placeholder, the jet patch of `0.orig/U`:

In [ ]:
text = (case / "0.orig" / "U").read_text()
print(text[text.index("    jet"):text.index("    inlet")])

`controller_params` builds what goes in there. The key `0.orig__U__controller` reads as: the variable `controller`, in the file `U`, in the folder `0.orig`. The block carries the policy path, the schedule, the bounds and the 99 probe positions, so the case cannot disagree with the network about where the observation is measured. `specHash` is a fingerprint of the spec; the solver copies it into the trajectory header and the reader refuses a mismatch.

In [ ]:
params = rl.controller_params(
    spec, root.resolve() / "policy.onnx", "0.orig__U__controller", seed=0
)

block = params["0.orig__U__controller"].splitlines()
print("\n".join(block[:9]))
print(f"    [{len(block) - 20} lines omitted, the 99 probe positions among them]")
print("\n".join(block[-11:]))

## 4. The reward

`forceCoeffs` writes $C_d$ and $C_l$ every CFD step; the trajectory has one row per control step. `align_to_control` averages the first onto the second, and `moving_average` then spans one shedding period, so the reward measures the effect of the action rather than the phase of the oscillation.

`CD_BASELINE` is the drag of the uncontrolled cylinder and only sets where zero reward sits. The value below was measured on the reference mesh of `cylinder2D_salehi`, averaged over three whole shedding periods; section 5 replaces it with the one from this mesh.

In [ ]:
SHEDDING_PERIOD = 5.8
LIFT_WEIGHT = 0.2
CD_BASELINE = 1.4007

window = int(round(SHEDDING_PERIOD / spec.control_interval))


def reward(ds):
    """Drag reduction against the uncontrolled baseline, penalized by lift."""
    cd = rl.moving_average(ds["Cd"], window)
    cl = rl.moving_average(ds["Cl"], window)
    return CD_BASELINE - (cd + LIFT_WEIGHT * np.abs(cl))


print(f"averaging window: {window} control steps")

On a synthetic force history, standing in for a policy that starts suppressing the shedding at $t = 10$. No simulation is involved here.

In [ ]:
t_cfd = np.arange(0.0, EPISODE_END + CFD_STEP, CFD_STEP)
rate = 2 * np.pi / SHEDDING_PERIOD
suppressed = np.clip((t_cfd - 10.0) / 20.0, 0.0, 1.0)

forces = xr.Dataset(
    {
        "Cd": ("time", 1.39 - 0.25 * suppressed
               + 0.04 * (1 - suppressed) * np.cos(2 * rate * t_cfd)),
        "Cl": ("time", 0.9 * (1 - suppressed) * np.sin(rate * t_cfd)),
    },
    coords={"time": t_cfd},
)

control_times = np.arange(1, n_control + 1) * spec.control_interval
aligned = rl.align_to_control(forces, control_times, how="mean")
reward = reward(aligned)

fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True, constrained_layout=True)
axes[0].plot(forces["time"], forces["Cd"], color="tab:gray", lw=1,
             label="every CFD step")
axes[0].plot(aligned["time"], rl.moving_average(aligned["Cd"], window), color="k",
             lw=1.5, label="averaged over a shedding period")
axes[0].set_ylabel("Cd")
axes[0].legend(frameon=False, fontsize=9)
axes[1].plot(control_times, reward, color="k", lw=1.5)
axes[1].axhline(0.0, color="gray", lw=0.8)
axes[1].set_ylabel("reward")
axes[1].set_xlabel("time")
plt.show()

print(f"return over the episode: {reward.sum():.1f}")

## 5. The uncontrolled baseline

`./Allwarmup` runs the case without control until the shedding saturates, writes that state into `0.orig` so every episode starts from it, and leaves its force history behind. The mean drag over a whole number of shedding periods of that history is the offset the reward needs.

`warmup/` is worth deleting once that number is recorded: everything the template holds is copied into every episode directory.

In [ ]:
warmup = case / "warmup"

if (warmup / "postProcessing" / "forces").is_dir():
    history = rl.read_function_object(warmup, "forces")
    settled = history.sel(time=slice(100.0, None))
    periods = int((float(settled["time"][-1]) - float(settled["time"][0]))
                  / SHEDDING_PERIOD)
    settled = settled.sel(
        time=slice(None, float(settled["time"][0]) + periods * SHEDDING_PERIOD)
    )
    CD_BASELINE = float(settled["Cd"].mean())
    print(f"Cd over {periods} shedding periods: {CD_BASELINE:.4f}, "
          f"against 1.4007 measured on the reference mesh")
else:
    print(f"no baseline yet: run ./Allwarmup in {case}")

## 6. One episode

One solver run is one episode. The runner renders the controller into the case, launches `Allrun`, reads the trajectory the solver wrote and the `forces` functionObject, aligns them and calls the reward.

In [ ]:
simulator = uqt.OpenFOAMSimulator(
    template_path=case,
    solver_script="Allrun",
    output_path=root,
    qoi_variables=["U", "p"],
)

runner = rl.ClosedLoopRunner(
    simulator,
    spec,
    salehi_reward,
    controller_keys="0.orig__U__controller",
    function_objects=["forces"],
)
runner

In [ ]:
def plot_episode(ds):
    fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True,
                             constrained_layout=True)
    axes[0].plot(ds["time"], ds["action"].isel(act_component=0),
                 drawstyle="steps-post", color="k", lw=1.2)
    axes[0].set_ylabel("jet velocity")
    axes[1].plot(ds["time"], ds["Cd"], color="k", lw=1.2)
    axes[1].set_ylabel("Cd")
    axes[1].set_xlabel("time")
    plt.show()


if have_controller:
    untrained = rl.export_random_policy(spec, root / "policy_untrained.onnx", seed=0)
    rollout = runner.collect(untrained, n_episodes=1)
    episode = rollout.episodes[0]
    print(f"{episode.sizes['time']} control steps, "
          f"return {float(episode['reward'].sum()):.2f}")
    plot_episode(episode)

## 7. Training

Each iteration exports one frozen policy, runs `n_episodes` cases with it, and updates on what they wrote. The hyperparameters are the published ones from `configs/agent.json`: two hidden layers of 64, learning rate $5 \times 10^{-4}$, 12 gradient epochs per update, clip $0.2$, discount $0.99$.

`n_episodes` and `n_jobs` are the width of one iteration. Six single core cases in parallel beat one case decomposed over six cores.

In [ ]:
ITERATIONS = 100
EPISODES_PER_ITERATION = 6

model = PPO(
    "MlpPolicy",
    runner,
    n_episodes=EPISODES_PER_ITERATION,
    n_jobs=EPISODES_PER_ITERATION,
    export_dir=root / "policies",
    policy_kwargs={"net_arch": [64, 64]},
    learning_rate=5e-4,
    batch_size=n_control,
    n_epochs=12,
    clip_range=0.2,
    gamma=0.99,
    verbose=1,
)

In [ ]:
def plot_returns(returns):
    fig, ax = plt.subplots(figsize=(8, 3), constrained_layout=True)
    ax.plot(returns, color="k", lw=1.2)
    ax.axhline(0.0, color="gray", lw=0.8)
    ax.set_xlabel("iteration")
    ax.set_ylabel("mean return")
    plt.show()


if have_controller:
    model.learn(total_timesteps=ITERATIONS * EPISODES_PER_ITERATION * n_control)
    plot_returns(model.returns_history)

## 8. Building the solver

The controlled episode needs `uqtopusPolicy`, which is `pimpleFoam` plus the `onnxPolicy` boundary condition. Once per machine.

**ONNX Runtime.** A prebuilt release, nothing to compile:

```bash
mkdir -p ~/opt && cd ~/opt
wget https://github.com/microsoft/onnxruntime/releases/download/v1.17.3/onnxruntime-linux-x64-1.17.3.tgz
tar xzf onnxruntime-linux-x64-1.17.3.tgz
export ONNXRUNTIME_ROOT=$HOME/opt/onnxruntime-linux-x64-1.17.3
```

Version 1.13 or newer, for opset 17 and for the metadata reader. The build sets an rpath, so `LD_LIBRARY_PATH` never has to be touched.

**The library.** The solver is never touched. Two classes compile into one shared object:

```bash
source /opt/openfoam9/etc/bashrc
cd uqtopusPolicy
wmake libso
```

`onnxPolicy` holds the ONNX Runtime session and answers "observation in, action out". `uqtopusController` decides once per control step and `uqtopusBoundaryCondition` writes one component of the action on its patch. The first has nothing to do with boundaries, so a well model or a source term can hold one too.

**Check.** `ls $FOAM_USER_LIBBIN/libuqtopusPolicy.so` should print a path, and the first cell of this notebook should now report it as found. The case already asks for it in `system/controlDict`:

```
application     pimpleFoam;
libs            ("libuqtopusPolicy.so");
```

`uqtopusPolicy/README.md` has the contract the class implements and the list of places most likely to need a fix on the first build, since it was written without a compiler at hand.